# Persistent Colab workspace

Keep files, a saved Python environment and system installation layers in Google Drive, then restore them on a compatible new runtime.

**Start:** choose CPU for ordinary development, or select a GPU when needed. On a fresh VM, run cells 1 and 2 before installing packages, changing system files or importing project dependencies, including the first time you use a project. This records the unmodified image baseline so saved changes and package rebuild recipes are accurate. Grant Drive access if Google asks. Use the same SPACE and PROJECT on future visits. Data lives under `My Drive/PersistentColabData/spaces/SPACE/workspaces/PROJECT`.

**Finish:** save model/training checkpoints, run **Save environment**, then **Flush Drive**. Finally use **Runtime → Disconnect and delete runtime** to release compute.

Drive preserves files and saved environment archives. A released VM loses RAM, GPU memory, running programs and unsaved changes. System layers require the same base image; changing images requires an explicit rebuild on a fresh VM. Host GPU drivers are not saved. Google Drive capacity does not enlarge the runtime disk.


In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. Restore workspace, system and Python environments
SPACE = 'default'  # @param {type:"string"}
PROJECT = 'default'  # @param {type:"string"}
exec('"""Restore a Python workspace from Drive without pretending the VM is persistent."""\nfrom __future__ import annotations\n\nimport hashlib\nimport gzip\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport site\nimport socket\nimport stat\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom datetime import datetime, timezone\nfrom contextlib import contextmanager\n\n\nSYSTEM_ROOTS = ("/usr", "/opt", "/etc", "/var/lib", "/root/.local")\nSYSTEM_EXCLUDES = (\n    "/proc", "/sys", "/dev", "/run", "/tmp", "/var/tmp", "/mnt", "/media", "/content",\n    "/root/.config", "/root/.cache", "/root/.ssh", "/root/.local/share/jupyter",\n    "/root/.local/share/keyrings", "/root/.local/share/colab", "/var/log",\n    "/var/cache/apt/archives", "/var/lib/apt/lists", "/var/lib/private",\n    "/etc/hostname", "/etc/hosts", "/etc/resolv.conf", "/etc/machine-id", "/etc/mtab",\n    "/etc/shadow", "/etc/shadow-", "/etc/gshadow", "/etc/gshadow-", "/etc/ssl/private",\n    "/etc/security/opasswd", "/var/lib/dbus/machine-id", "/var/lib/dpkg/lock",\n    "/var/lib/dpkg/lock-frontend", "/opt/colab-workspace",\n    "/usr/bin/rclone", "/usr/local/bin/rclone", "/etc/pip.conf", "/etc/apt/auth.conf",\n    "/etc/apt/auth.conf.d",\n)\n\n\nclass PhaseTimer:\n    """Wall-clock seconds for non-overlapping operation phases."""\n\n    def __init__(self):\n        self.started = time.monotonic()\n        self.phases = {}\n\n    @contextmanager\n    def phase(self, name):\n        started = time.monotonic()\n        try:\n            yield\n        finally:\n            self.phases[name] = round(time.monotonic() - started, 3)\n\n    def metrics(self):\n        return {"seconds": round(time.monotonic() - self.started, 3),\n                "phases": dict(self.phases)}\n\n\ndef valid_name(value, label="Project"):\n    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", value):\n        raise ValueError(f"{label} must use letters, digits, underscores or hyphens.")\n    return value\n\n\ndef container_start():\n    """PID 1 start in wall-clock seconds (host boot alone is not the container start)."""\n    boot = int(next(line.split()[1] for line in Path("/proc/stat").read_text().splitlines()\n                    if line.startswith("btime ")))\n    ticks = int(Path("/proc/1/stat").read_text().rsplit(")", 1)[1].split()[19])\n    return boot + ticks / os.sysconf("SC_CLK_TCK")\n\n\ndef system_image():\n    """Use immutable image markers, never the mutable installed-package list."""\n    release = os.environ.get("COLAB_RELEASE_TAG")\n    if not release:\n        raise RuntimeError("System persistence requires a Colab image release identifier.")\n    cache = Path("/content/persistent-colab/system-image.json")\n    started = container_start()\n    if cache.exists():\n        value = json.loads(cache.read_text())\n        if value["container_start"] == started:\n            return value["id"], value["descriptor"]\n    cuda = Path("/usr/local/cuda/version.json")\n    descriptor = {"release": release, "python": platform.python_version(),\n                  "machine": platform.machine(), "libc": list(platform.libc_ver()),\n                  "os_release": Path("/etc/os-release").read_text(),\n                  "accelerator_image": bool(os.environ.get("COLAB_GPU", "").strip("0")),\n                  "cuda": cuda.read_text() if cuda.exists() else None}\n    image_id = hashlib.sha256(json.dumps(descriptor, sort_keys=True).encode()).hexdigest()[:24]\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    write_json(cache, {"container_start": started, "id": image_id, "descriptor": descriptor})\n    return image_id, descriptor\n\n\ndef system_packages():\n    apt = subprocess.check_output(["dpkg-query", "-W", "-f=${binary:Package}\\t${Version}\\t${db:Status-Status}\\n"], text=True)\n    apt = dict((fields[0], fields[1]) for line in apt.splitlines()\n               if len(fields := line.split("\\t")) == 3 and fields[2] == "installed")\n    pip = json.loads(subprocess.check_output([sys.executable, "-m", "pip", "list", "--format=json"], text=True))\n    return {"apt": apt, "pip": {p["name"].lower().replace("_", "-"): p["version"] for p in pip}}\n\n\ndef _write_gzip_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with gzip.open(temporary, "wt", compresslevel=1) as stream:\n        json.dump(value, stream, separators=(",", ":"))\n    with temporary.open("rb") as stream:\n        os.fsync(stream.fileno())\n    os.replace(temporary, path)\n\n\ndef _read_gzip_json(path):\n    with gzip.open(path, "rt") as stream:\n        return json.load(stream)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with temporary.open("w") as stream:\n        json.dump(value, stream, indent=2)\n        stream.write("\\n")\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temporary, path)\n\n\ndef runtime_signature():\n    # Base packages are inherited, rather than copied into each environment.\n    # A changed base image requires an explicit rebuild, never a silent restore.\n    packages = json.loads(subprocess.check_output(\n        [sys.executable, "-m", "pip", "list", "--format=json"], text=True))\n    normalized = sorted((p["name"].lower().replace("_", "-"), p["version"])\n                        for p in packages)\n    return {\n        "python": list(sys.version_info[:2]),\n        "machine": platform.machine(),\n        "os": platform.system(),\n        "libc": list(platform.libc_ver()),\n        "base_packages_sha256": hashlib.sha256(\n            json.dumps(normalized).encode()).hexdigest(),\n        "base_packages": [list(item) for item in normalized],\n    }\n\n\nPLATFORM_KEYS = ("python", "machine", "os", "libc")\n\n\ndef package_drift(saved, current):\n    """Describe base package differences as sorted \'name: old -> new\' lines."""\n    before, after = dict(map(tuple, saved)), dict(map(tuple, current))\n    return [f"{name}: {before.get(name, \'absent\')} -> {after.get(name, \'absent\')}"\n            for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]\n\n\nLOCK_HEARTBEAT = 60\nLOCK_STALE = 600\n\n\ndef _utc():\n    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n\n\ndef _lock_age(record):\n    heartbeat = datetime.strptime(record["heartbeat_utc"], "%Y-%m-%dT%H:%M:%SZ")\n    return (datetime.now(timezone.utc) - heartbeat.replace(tzinfo=timezone.utc)).total_seconds()\n\n\ndef _heartbeat_name(path):\n    return "colab-workspace-lock:" + str(path)\n\n\ndef _start_heartbeat(path):\n    name = _heartbeat_name(path)\n    if any(thread.name == name and thread.is_alive() for thread in threading.enumerate()):\n        return\n    stop = threading.Event()\n\n    def beat():\n        while not stop.wait(LOCK_HEARTBEAT):\n            try:\n                record = json.loads(path.read_text())\n                if record.get("host") != socket.gethostname():\n                    return\n                record.update(heartbeat_utc=_utc(), pid=os.getpid())\n                write_json(path, record)\n            except (OSError, ValueError):\n                # Storage was unmounted or the lock was removed; never recreate it.\n                return\n\n    thread = threading.Thread(target=beat, name=name, daemon=True)\n    thread.stop = stop\n    thread.start()\n\n\ndef acquire_lock(root, session=None, force=False):\n    """Claim a workspace for this runtime; detects, but cannot prevent, racing writers."""\n    path = Path(root) / "environment" / "LOCK"\n    host = socket.gethostname()\n    record = {"session": session, "host": host, "pid": os.getpid(),\n              "created_utc": _utc(), "heartbeat_utc": _utc()}\n    try:\n        with path.open("xb") as stream:\n            stream.write(json.dumps(record, indent=2).encode() + b"\\n")\n            stream.flush()\n            os.fsync(stream.fileno())\n    except FileExistsError:\n        try:\n            current = json.loads(path.read_text())\n        except (OSError, ValueError):\n            current = None\n        if not force and (current is None or current.get("host") != host):\n            if current is None:\n                raise RuntimeError(f"Unreadable workspace lock {path}. Retry with --force-unlock "\n                                   "(notebook: ws.prepare(force_unlock=True)) if no runtime uses it.") from None\n            age = _lock_age(current)\n            owner = f"session {current.get(\'session\')!r} on host {current.get(\'host\')!r}"\n            if age > LOCK_STALE:\n                raise RuntimeError(f"Workspace lock held by {owner} is stale (no heartbeat for {age / 60:.0f} min). "\n                                   "If that runtime is gone, retry with --force-unlock "\n                                   "(notebook: ws.prepare(force_unlock=True)).") from None\n            raise RuntimeError(f"Workspace is in use by {owner} (heartbeat {age:.0f}s ago). "\n                               "Use one writer per project; finish or flush the other runtime first.") from None\n        if current and current.get("host") == host:\n            record["created_utc"] = current.get("created_utc", record["created_utc"])\n        write_json(path, record)\n    _start_heartbeat(path)\n    return path\n\n\ndef lock_held_elsewhere(root):\n    path = Path(root) / "environment" / "LOCK"\n    try:\n        return json.loads(path.read_text()).get("host") != socket.gethostname()\n    except FileNotFoundError:\n        return False\n    except ValueError:\n        return True\n\n\ndef release_lock(root):\n    """Stop the heartbeat and remove this runtime\'s lock before storage is flushed."""\n    path = Path(root) / "environment" / "LOCK"\n    for thread in threading.enumerate():\n        if thread.name == _heartbeat_name(path):\n            thread.stop.set()\n            thread.join()\n    if path.exists() and not lock_held_elsewhere(root):\n        state = rclone_state()\n        if state and path.resolve().is_relative_to(Path(state["mount_point"]).resolve()):\n            # Deleting a file whose rewrite is still queued drops only the queued\n            # upload; the previous object would stay on the remote.\n            wait_rclone_uploads(state)\n        path.unlink()\n\n\ndef prune_snapshots(root, keep, apply=False):\n    """Remove old snapshot pairs, always keeping the newest ones and the latest pointer."""\n    if keep < 1:\n        raise ValueError("keep must be at least 1.")\n    snapshots = Path(root) / "environment" / "snapshots"\n    latest = Path(root) / "environment" / "latest.json"\n    current = json.loads(latest.read_text())["created_utc"] if latest.exists() else None\n    stamps = sorted({path.name.split(".", 1)[0] for path in snapshots.iterdir()\n                     if path.name.endswith((".tar.gz", ".json"))}, reverse=True)\n    retained = set(stamps[:keep]) | {current}\n    removed = [stamp for stamp in stamps if stamp not in retained]\n    for stamp in removed:\n        for path in (snapshots / (stamp + ".tar.gz"), snapshots / (stamp + ".json")):\n            if path.exists():\n                print(("Deleted " if apply else "Would delete ") + str(path))\n                if apply:\n                    path.unlink()\n    print(f"Kept {len(stamps) - len(removed)} snapshot(s); {\'deleted\' if apply else \'would delete\'} {len(removed)}.")\n    if removed and not apply:\n        print("Dry run only. Repeat with --yes to delete.")\n    _prune_system_layers(Path(root), keep, apply, retained)\n    return removed\n\n\ndef _prune_system_layers(root, keep, apply, retained_snapshots):\n    protected = {}\n    pointers = list((root / "environment/system").glob("*/latest.json"))\n    for stamp in retained_snapshots - {None}:\n        path = root / "environment/snapshots" / (stamp + ".json")\n        if path.exists():\n            snapshot = json.loads(path.read_text())\n            if snapshot.get("system"):\n                pointer = snapshot["system"]\n                protected.setdefault(pointer["image_id"], set()).update(item["id"] for item in pointer["chain"])\n    for path in pointers:\n        pointer = json.loads(path.read_text())\n        protected.setdefault(pointer["image_id"], set()).update(item["id"] for item in pointer["chain"])\n    for folder in (root / "environment/system").glob("*/layers"):\n        ids = sorted({path.name.split(".", 1)[0] for path in folder.glob("*.json")}, reverse=True)\n        retained = protected.get(folder.parent.name, set()) | set(ids[:keep])\n        for layer_id in set(ids) - retained:\n            for extension in (".json", ".tar.gz"):\n                path = folder / (layer_id + extension)\n                print(("Deleted " if apply else "Would delete ") + str(path))\n                if apply:\n                    path.unlink(missing_ok=True)\n\n\nRCLONE_CONF = Path("/root/.config/rclone/rclone.conf")\nRCLONE_STATE = Path("/root/.config/colab-workspace/rclone-mount.json")\nRCLONE_RC = "127.0.0.1:5572"\nRCLONE_LOG = Path("/tmp/colab-workspace-rclone.log")\n# Colab ships FUSE 2 only; rclone execs fusermount3, which accepts the same protocol.\nFUSE_SHIM = Path("/opt/colab-workspace/bin")\n\n\ndef _rclone(*arguments, config=RCLONE_CONF):\n    return subprocess.run(["rclone", "--config", str(config), *map(str, arguments)],\n                          check=True, capture_output=True, text=True, stdin=subprocess.DEVNULL)\n\n\ndef _remote_join(remote, relative):\n    relative = str(relative).strip("/")\n    if not relative or relative == ".":\n        return remote\n    return remote + relative if remote.endswith(":") else remote + "/" + relative\n\n\ndef rclone_state():\n    return json.loads(RCLONE_STATE.read_text()) if RCLONE_STATE.exists() else None\n\n\ndef _install_rclone():\n    if shutil.which("rclone"):\n        return\n    script = subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True,\n                            capture_output=True, text=True, stdin=subprocess.DEVNULL)\n    if script.returncode and not shutil.which("rclone"):\n        apt = subprocess.run("apt-get update -qq && apt-get install -y -qq rclone", shell=True,\n                             capture_output=True, text=True, stdin=subprocess.DEVNULL)\n        if apt.returncode:\n            raise RuntimeError("Could not install rclone:\\n" + script.stderr[-1000:] + apt.stderr[-1000:])\n\n\ndef _fuse_environment():\n    environment = os.environ.copy()\n    if not shutil.which("fusermount3"):\n        fusermount = shutil.which("fusermount")\n        if fusermount is None or not Path("/dev/fuse").exists():\n            raise RuntimeError("FUSE is unavailable on this runtime (/dev/fuse or fusermount missing).")\n        FUSE_SHIM.mkdir(parents=True, exist_ok=True)\n        shim = FUSE_SHIM / "fusermount3"\n        if not shim.is_symlink():\n            shim.symlink_to(fusermount)\n        environment["PATH"] = str(FUSE_SHIM) + os.pathsep + environment["PATH"]\n    return environment\n\n\ndef mount_rclone(remote, mount_point, staged_config, timeout=120):\n    """Install a staged rclone config and mount the remote; remove the config on failure."""\n    mount_point = Path(mount_point).resolve()\n    state = rclone_state()\n    if os.path.ismount(mount_point):\n        Path(staged_config).unlink(missing_ok=True)\n        if state and state["remote"] == remote and state["mount_point"] == str(mount_point):\n            print("rclone remote is already mounted:", mount_point)\n            return state\n        raise RuntimeError(f"{mount_point} is already mounted by something else.")\n    try:\n        RCLONE_CONF.parent.mkdir(parents=True, exist_ok=True)\n        os.chmod(RCLONE_CONF.parent, 0o700)\n        shutil.move(staged_config, RCLONE_CONF)\n        os.chmod(RCLONE_CONF, 0o600)\n        _install_rclone()\n        environment = _fuse_environment()\n        mount_point.mkdir(parents=True, exist_ok=True)\n        if any(mount_point.iterdir()):\n            raise RuntimeError(f"{mount_point} is not empty; choose an empty mount point.")\n        # A new session keeps the daemon alive when the kernel restarts.\n        daemon = subprocess.run(\n            ["rclone", "--config", str(RCLONE_CONF), "mount", remote, str(mount_point),\n             "--vfs-cache-mode", "writes", "--rc", "--rc-addr", RCLONE_RC, "--rc-no-auth",\n             "--daemon", "--log-file", str(RCLONE_LOG), "--log-level", "NOTICE"],\n            env=environment, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL,\n            stderr=subprocess.DEVNULL, start_new_session=True)\n        deadline = time.monotonic() + timeout\n        while not os.path.ismount(mount_point):\n            if daemon.returncode or time.monotonic() > deadline:\n                log = RCLONE_LOG.read_text()[-2000:] if RCLONE_LOG.exists() else ""\n                raise RuntimeError("rclone mount did not become ready:\\n" + log)\n            time.sleep(1)\n        os.listdir(mount_point)\n        state = {"remote": remote, "mount_point": str(mount_point),\n                 "config": str(RCLONE_CONF), "rc": RCLONE_RC}\n        RCLONE_STATE.parent.mkdir(parents=True, exist_ok=True)\n        write_json(RCLONE_STATE, state)\n        print("Mounted rclone remote at", mount_point)\n        return state\n    except BaseException:\n        Path(staged_config).unlink(missing_ok=True)\n        RCLONE_CONF.unlink(missing_ok=True)\n        raise\n\n\ndef _md5(path):\n    digest = hashlib.md5()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef wait_rclone_uploads(state, timeout=1800, poll=2):\n    """Block until the rclone VFS has no queued or running uploads."""\n    url = "http://" + state["rc"] + "/"\n    deadline = time.monotonic() + timeout\n    while True:\n        cache = json.loads(_rclone("rc", "vfs/stats", "--url", url, config=state["config"]).stdout)["diskCache"]\n        if cache.get("erroredFiles"):\n            raise RuntimeError(f"rclone reports {cache[\'erroredFiles\']} file(s) that failed to upload; see {RCLONE_LOG}.")\n        if cache["uploadsInProgress"] == 0 and cache["uploadsQueued"] == 0:\n            return\n        if time.monotonic() > deadline:\n            raise RuntimeError("Timed out waiting for rclone uploads; the remote is still mounted.")\n        time.sleep(poll)\n\n\ndef flush_rclone(workspace_roots=(), timeout=1800, poll=2):\n    """Wait for VFS uploads, verify the remote copy, unmount and delete the config."""\n    state = rclone_state()\n    if state is None:\n        raise RuntimeError("No rclone mount was created by colab-workspace on this runtime.")\n    mount_point = Path(state["mount_point"]).resolve()\n    config = Path(state["config"])\n    deadline = time.monotonic() + timeout\n    wait_rclone_uploads(state, timeout, poll)\n    for root in map(Path, workspace_roots):\n        remote_root = _remote_join(state["remote"], root.relative_to(mount_point))\n        remote_files = _rclone("lsf", "--files-only", _remote_join(remote_root, "environment"),\n                               config=config).stdout.split("\\n")\n        if "LOCK" in remote_files:\n            remote_lock = json.loads(_rclone("cat", _remote_join(remote_root, "environment/LOCK"),\n                                             config=config).stdout)\n            if remote_lock.get("host") == socket.gethostname():\n                raise RuntimeError("This runtime\'s lock is still on the remote; the remote is still mounted.")\n        _verify_rclone_system(state, root)\n        latest = root / "environment" / "latest.json"\n        if not latest.exists():\n            continue\n        local_pointer = json.loads(latest.read_text())\n        remote_pointer = json.loads(_rclone(\n            "cat", _remote_join(remote_root, "environment/latest.json"), config=config).stdout)\n        if remote_pointer != local_pointer:\n            raise RuntimeError("Remote latest.json differs from the local copy; the remote is still mounted.")\n        archive = root / "environment" / "snapshots" / local_pointer["archive"]\n        listing = json.loads(_rclone(\n            "lsjson", "--stat", "--hash", "--hash-type", "md5",\n            _remote_join(remote_root, "environment/snapshots/" + local_pointer["archive"]),\n            config=config).stdout)\n        remote_md5 = (listing.get("Hashes") or {}).get("md5")\n        if listing.get("Size") != archive.stat().st_size or (remote_md5 and remote_md5 != _md5(archive)):\n            raise RuntimeError("Remote snapshot archive does not match; the remote is still mounted.")\n        print("Verified remote snapshot:", local_pointer["archive"])\n    if Path.cwd().is_relative_to(mount_point):\n        os.chdir("/")\n    fusermount = shutil.which("fusermount3") or shutil.which("fusermount")\n    if fusermount is None:\n        raise RuntimeError("fusermount is unavailable; the remote is still mounted.")\n    subprocess.run([fusermount, "-u", str(mount_point)], check=True,\n                   capture_output=True, text=True, stdin=subprocess.DEVNULL)\n    while os.path.ismount(mount_point):\n        if time.monotonic() > deadline:\n            raise RuntimeError("Timed out unmounting the rclone remote.")\n        time.sleep(poll)\n    # rclone may rewrite the config when refreshing a token; remove every copy.\n    for path in config.parent.glob(config.name + "*"):\n        path.unlink()\n    RCLONE_STATE.unlink()\n    print("rclone uploads verified; remote unmounted and credentials removed.")\n\n\ndef _verify_rclone_system(state, root):\n    """Verify system-only rebuilds too, even before a venv snapshot exists."""\n    config = state["config"]\n    remote_root = _remote_join(state["remote"], root.relative_to(Path(state["mount_point"])))\n    # System layers are part of finish\'s durability gate, including the image baseline.\n    for system_latest in (root / "environment/system").glob("*/latest.json"):\n        relative = system_latest.relative_to(root)\n        system_pointer = json.loads(system_latest.read_text())\n        remote_system = json.loads(_rclone("cat", _remote_join(remote_root, relative), config=config).stdout)\n        if remote_system != system_pointer:\n            raise RuntimeError("Remote system pointer differs; the remote is still mounted.")\n        for reference in system_pointer["chain"]:\n            manifest_path = system_latest.parent / "layers" / (reference["id"] + ".json")\n            manifest = json.loads(manifest_path.read_text())\n            remote_manifest = _rclone("cat", _remote_join(remote_root, manifest_path.relative_to(root)),\n                                      config=config).stdout\n            if json.loads(remote_manifest) != manifest or sha256(manifest_path) != reference["sha256"]:\n                raise RuntimeError("Remote system manifest differs; the remote is still mounted.")\n            _verify_rclone_file(state, manifest_path.with_suffix(".tar.gz"))\n        data_root = root.parents[3]\n        baseline = data_root / "images" / system_pointer["image_id"] / "baseline.json.gz"\n        if sha256(baseline) != system_pointer["baseline_sha256"]:\n            raise RuntimeError("System baseline differs; the remote is still mounted.")\n        _verify_rclone_file(state, baseline)\n        _verify_rclone_file(state, baseline.with_name("baseline.json"))\n        print("Verified remote system chain:", system_pointer["image_id"], len(system_pointer["chain"]))\n\n\ndef _verify_rclone_file(state, path):\n    remote = _remote_join(state["remote"], path.relative_to(Path(state["mount_point"])))\n    listing = json.loads(_rclone("lsjson", "--stat", "--hash", "--hash-type", "md5", remote,\n                                config=state["config"]).stdout)\n    md5 = (listing.get("Hashes") or {}).get("md5")\n    if listing.get("Size") != path.stat().st_size or (md5 and md5 != _md5(path)):\n        raise RuntimeError("Remote system archive does not match; the remote is still mounted.")\n\n\nclass SystemLayer:\n    """Immutable, image-specific filesystem deltas; restore never invokes an installer.\n\n    The filesystem and package inventory are injectable for account-free tests.\n    Production callers must initialize on a fresh VM before installing packages.\n    """\n\n    def __init__(self, workspace_root, data_root, local, *, fs_root="/", image_id=None,\n                 start_time=None, packages=None, mounts=None, owner_path=None):\n        self.root = Path(workspace_root)\n        self.data_root = Path(data_root)\n        self.local = Path(local)\n        self.fs = Path(fs_root).resolve()\n        self.real = self.fs == Path("/")\n        self.image_id, self.image = system_image() if image_id is None else (image_id, {"test_image": image_id})\n        valid_name(self.image_id, "Image ID")\n        self.start = container_start() if start_time is None else start_time\n        self.inventory = packages or system_packages\n        self.folder = self.root / "environment" / "system" / self.image_id\n        self.layers = self.folder / "layers"\n        self.latest = self.folder / "latest.json"\n        self.baseline_dir = self.data_root / "images" / self.image_id\n        self.baseline_path = self.baseline_dir / "baseline.json.gz"\n        self.state_path = self.local / "system-state.json.gz"\n        self.owner_path = Path(owner_path or ("/content/persistent-colab/system-owner.json" if self.real\n                                              else self.fs / "content/system-owner.json"))\n        if mounts is None:\n            mounts = [line.split()[4].replace("\\\\040", " ").replace("\\\\011", "\\t").replace("\\\\134", "\\\\")\n                      for line in Path("/proc/self/mountinfo").read_text().splitlines()] if self.real else []\n        self.mounts = {p.rstrip("/") for p in mounts if p != "/"}\n        self.device = self.fs.stat().st_dev\n        self.baseline = None\n\n    def allowed(self, name):\n        parts = Path(name).parts\n        return (name.startswith("/") and ".." not in parts and "__pycache__" not in parts\n                and parts[-1] != "direct_url.json"\n                and any(name == root or name.startswith(root + "/") for root in SYSTEM_ROOTS)\n                and not any(name == root or name.startswith(root + "/") for root in SYSTEM_EXCLUDES)\n                and not name.startswith("/etc/ssh/ssh_host_")\n                and not any(name == root or name.startswith(root + "/") for root in self.mounts))\n\n    def path(self, name):\n        if not self.allowed(name):\n            raise RuntimeError("Excluded or invalid system path: " + name)\n        return self.fs / name.lstrip("/")\n\n    @staticmethod\n    def fingerprint(info, path):\n        return [info.st_ctime_ns, info.st_mtime_ns, info.st_size, info.st_mode,\n                info.st_uid, info.st_gid, os.readlink(path) if stat.S_ISLNK(info.st_mode) else None]\n\n    def scan(self):\n        """Walk only the root device, never following links or entering bind mounts."""\n        result = {}\n        def visit(path, name):\n            if not self.allowed(name):\n                return\n            try:\n                info = path.lstat()\n            except FileNotFoundError:\n                return\n            if info.st_dev != self.device or not (stat.S_ISREG(info.st_mode) or\n                    stat.S_ISDIR(info.st_mode) or stat.S_ISLNK(info.st_mode)):\n                return\n            result[name] = self.fingerprint(info, path)\n            if stat.S_ISDIR(info.st_mode):\n                with os.scandir(path) as entries:\n                    for entry in entries:\n                        visit(Path(entry.path), name + "/" + entry.name)\n        for root in SYSTEM_ROOTS:\n            visit(self.fs / root.lstrip("/"), root)\n        return result\n\n    def ensure_baseline(self):\n        if self.baseline_path.exists():\n            meta = json.loads((self.baseline_dir / "baseline.json").read_text())\n            self.baseline_sha256 = sha256(self.baseline_path)\n            if self.baseline_sha256 != meta["sha256"]:\n                raise RuntimeError("System baseline checksum mismatch.")\n            self.baseline = _read_gzip_json(self.baseline_path)\n        else:\n            # This must happen during prepare on the fresh image, before user installs.\n            started = time.monotonic()\n            paths = self.scan()\n            value = {"schema": 1, "image_id": self.image_id, "image": self.image,\n                     "paths": sorted(paths), "packages": self.inventory()}\n            self.baseline_dir.mkdir(parents=True, exist_ok=True)\n            self.local.mkdir(parents=True, exist_ok=True)\n            with tempfile.TemporaryDirectory(dir=self.local) as directory:\n                source = Path(directory) / "baseline.json.gz"\n                _write_gzip_json(source, value)\n                self.baseline_sha256 = sha256(source)\n                self._copy_verified(source, self.baseline_path, self.baseline_sha256)\n            write_json(self.baseline_dir / "baseline.json", {"image_id": self.image_id,\n                       "sha256": self.baseline_sha256})\n            self.baseline = value\n            print(f"System baseline: {len(paths)} paths, {self.baseline_path.stat().st_size} bytes, "\n                  f"{time.monotonic() - started:.2f}s; image {self.image_id}.")\n        if self.baseline["image_id"] != self.image_id:\n            raise RuntimeError("System baseline image ID mismatch.")\n        return self.baseline\n\n    @staticmethod\n    def _copy_verified(source, destination, expected=None):\n        expected = expected or sha256(source)\n        with Path(source).open("rb") as incoming, Path(destination).open("xb") as outgoing:\n            shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)\n            outgoing.flush()\n            os.fsync(outgoing.fileno())\n        if sha256(destination) != expected:\n            raise RuntimeError("System layer readback failed; previous pointer remains current.")\n\n    def _claim(self):\n        identity = {"workspace": str(self.root.resolve()), "image_id": self.image_id}\n        self.owner_path.parent.mkdir(parents=True, exist_ok=True)\n        try:\n            with self.owner_path.open("x") as stream:\n                json.dump(identity, stream)\n                stream.flush()\n                os.fsync(stream.fileno())\n        except FileExistsError:\n            owner = json.loads(self.owner_path.read_text())\n            if owner != identity:\n                raise RuntimeError("Another project\'s system layer is already applied on this VM. "\n                                   "Use a fresh VM for " + str(self.root) + ".")\n\n    def _load_pointer(self):\n        if self.latest.exists():\n            pointer = json.loads(self.latest.read_text())\n            if pointer.get("image_id") != self.image_id:\n                raise RuntimeError("System layer image ID mismatch; no files were restored.")\n            return pointer\n        alternatives = sorted((self.root / "environment/system").glob("*/latest.json"))\n        if alternatives:\n            recipes = [json.loads(path.read_text()) for path in alternatives]\n            lines = ["System image ID mismatch; no files were restored. Current image: " + self.image_id]\n            for recipe in recipes:\n                lines.append("Saved image " + recipe["image_id"] + ": " + json.dumps(recipe["packages"], sort_keys=True))\n            raise RuntimeError("\\n".join(lines) + "\\nUse a fresh VM and explicitly run ws.rebuild_system() "\n                               "(CLI: rebuild-system) to reinstall recorded apt/system pip packages.")\n        return None\n\n    def _chain(self, pointer):\n        if pointer.get("image_id") != self.image_id:\n            raise RuntimeError("System layer image ID mismatch; no files were restored.")\n        result = []\n        seen = set()\n        for reference in pointer["chain"]:\n            layer_id = valid_name(reference["id"], "Layer ID")\n            if layer_id in seen:\n                raise RuntimeError("Repeated system layer in chain.")\n            seen.add(layer_id)\n            path = self.layers / (layer_id + ".json")\n            contents = path.read_bytes()\n            if hashlib.sha256(contents).hexdigest() != reference["sha256"]:\n                raise RuntimeError("System manifest checksum mismatch; no files were restored.")\n            manifest = json.loads(contents)\n            if manifest["image_id"] != self.image_id or manifest["id"] != layer_id:\n                raise RuntimeError("System manifest image ID mismatch; no files were restored.")\n            if manifest.get("parent") != (result[-1][0]["id"] if result else None):\n                raise RuntimeError("Broken system layer chain.")\n            result.append((manifest, self.layers / (layer_id + ".tar.gz")))\n        return result\n\n    def _safe_destination(self, name):\n        target = self.path(name)\n        current = target.parent\n        while current != self.fs:\n            if current.is_symlink():\n                raise RuntimeError("System archive crosses a symlink parent: " + name)\n            current = current.parent\n        return target\n\n    def _validate_archive(self, archive, manifest):\n        names = set()\n        links = set()\n        for member in archive.getmembers():\n            name = "/" + member.name\n            if member.name.startswith("/") or name not in manifest["files"] or not self.allowed(name):\n                raise RuntimeError("Invalid system archive member: " + member.name)\n            if name in names or not (member.isfile() or member.isdir() or member.issym()):\n                raise RuntimeError("Unsupported system archive member: " + member.name)\n            names.add(name)\n            if member.issym():\n                # Preserve user-space links, including absolute /etc/alternatives links.\n                # They are never followed while extracting other members.\n                links.add(name)\n            self._safe_destination(name)\n        if names != set(manifest["files"]):\n            raise RuntimeError("System archive and manifest paths differ.")\n        if any(any(parent in links for parent in map(str, Path(name).parents)) for name in names):\n            raise RuntimeError("System archive contains a symlink parent.")\n        for name in manifest["deleted"]:\n            self._safe_destination(name)\n\n    def _apply(self, archive, manifest):\n        # Delete deepest paths first. rmdir never removes an unrecorded child.\n        for name in sorted(manifest["deleted"], key=lambda p: (p.count("/"), p), reverse=True):\n            path = self._safe_destination(name)\n            if path.is_symlink() or path.is_file():\n                path.unlink()\n            elif path.exists():\n                path.rmdir()\n        for member in archive.getmembers():\n            path = self._safe_destination("/" + member.name)\n            if path.is_symlink() or (path.exists() and not member.isdir()):\n                if path.is_dir() and not path.is_symlink():\n                    path.rmdir()\n                else:\n                    path.unlink()\n            elif path.exists() and member.isdir() and not path.is_dir():\n                path.unlink()\n            # All names, types and symlink parents were validated before any writes.\n            archive.extract(member, path=self.fs, filter="fully_trusted")\n\n    def _restored_paths(self, chain):\n        """Derive image paths and stat only paths written by the verified chain.\n\n        None denotes an untouched image path. On the next full save scan, its\n        ctime must precede the restore checkpoint to remain unchanged. ctime\n        detects content, metadata and link changes even if mtime is preserved.\n        Unknown runtime paths are new, and missing image paths are deletions.\n        """\n        excluded = re.compile(r"^(?:" + "|".join(map(re.escape, SYSTEM_EXCLUDES + tuple(self.mounts))) + r")(?:/|$)")\n        paths = dict.fromkeys(name for name in self.baseline["paths"] if not excluded.match(name))\n        written = set()\n        for manifest, _ in chain:\n            for name in manifest["deleted"]:\n                paths.pop(name, None)\n            written.difference_update(manifest["deleted"])\n            written.update(manifest["files"])\n        for name in written:\n            path = self._safe_destination(name)\n            info = path.lstat()\n            paths[name] = self.fingerprint(info, path)\n        return paths\n\n    def restore(self):\n        timer = PhaseTimer()\n        if (self.local / "system-restore-failed.json").exists():\n            raise RuntimeError("System restore failed on this VM. Start a fresh VM before restoring again.")\n        with timer.phase("pointer_claim"):\n            pointer = self._load_pointer()\n            self._claim()\n        with timer.phase("baseline"):\n            self.ensure_baseline()\n            if pointer and pointer["baseline_sha256"] != self.baseline_sha256:\n                raise RuntimeError("System baseline does not match the saved chain; no files were restored.")\n        if self.state_path.exists():\n            with timer.phase("local_state"):\n                state = _read_gzip_json(self.state_path)\n            if state["image_id"] == self.image_id and state.get("chain") == (pointer or {}).get("chain", []):\n                write_json(self.local / "system-restore.json", {"image_id": self.image_id,\n                           "layers": 0, "bytes": 0, "already_applied": True, **timer.metrics()})\n                return pointer\n            raise RuntimeError("Local system state differs from the saved chain. Use a fresh VM.")\n        size = 0\n        with timer.phase("manifests"):\n            chain = self._chain(pointer) if pointer else []\n        # Validate the complete chain and copy each archive locally before touching /usr.\n        self.local.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(dir=self.local) as directory:\n            pending = []\n            with timer.phase("copy_verify_validate"):\n                for manifest, source in chain:\n                    target = Path(directory) / source.name\n                    shutil.copyfile(source, target)\n                    if sha256(target) != manifest["sha256"]:\n                        raise RuntimeError("System layer checksum mismatch; no files were restored.")\n                    with tarfile.open(target, "r:gz") as archive:\n                        self._validate_archive(archive, manifest)\n                    size += target.stat().st_size\n                    pending.append((manifest, target))\n            try:\n                with timer.phase("apply"):\n                    for manifest, path in pending:\n                        with tarfile.open(path, "r:gz") as archive:\n                            self._apply(archive, manifest)\n                with timer.phase("ldconfig"):\n                    if chain and self.real:\n                        subprocess.run(["ldconfig"], check=True, capture_output=True)\n            except BaseException:\n                # A partial root-filesystem restore cannot be rolled back safely in place.\n                write_json(self.local / "system-restore-failed.json", {"image_id": self.image_id})\n                raise\n        # Changes after this checkpoint are checked by the next save\'s full scan.\n        baseline_before_ns = time.time_ns()\n        with timer.phase("state_derive"):\n            paths = self._restored_paths(chain)\n        with timer.phase("state_write"):\n            self._write_state(paths, pointer, baseline_before_ns=baseline_before_ns)\n        metrics = timer.metrics()\n        duration = metrics["seconds"]\n        write_json(self.local / "system-restore.json", {"image_id": self.image_id,\n                   "layers": len(chain), "bytes": size, **metrics})\n        print(f"System restored: {len(chain)} layer(s), {size} compressed bytes, "\n              f"{duration:.2f}s; image {self.image_id}.")\n        print("System restore phases (seconds):", json.dumps(metrics["phases"], sort_keys=True))\n        return pointer\n\n    def _write_state(self, paths, pointer, baseline_before_ns=None, destination=None):\n        _write_gzip_json(destination or self.state_path, {"image_id": self.image_id,\n                         "chain": (pointer or {}).get("chain", []), "paths": paths,\n                         "baseline_before_ns": baseline_before_ns})\n\n    def _package_changes(self):\n        current = self.inventory()\n        return {kind: {name: version for name, version in current[kind].items()\n                       if self.baseline["packages"][kind].get(name) != version} for kind in ("apt", "pip")}\n\n    def save(self, compact=False):\n        timer = PhaseTimer()\n        with timer.phase("baseline_claim"):\n            self._claim()\n            self.ensure_baseline()\n        if (self.local / "system-restore-failed.json").exists():\n            raise RuntimeError("System restore failed on this VM. Use a fresh VM; saving is disabled.")\n        if not self.state_path.exists():\n            raise RuntimeError("Prepare system persistence on a fresh VM before saving.")\n        with timer.phase("previous_state"):\n            previous = _read_gzip_json(self.state_path)\n            pointer = None if getattr(self, "migrating", False) and not self.latest.exists() else self._load_pointer()\n            if previous.get("chain") != (pointer or {}).get("chain", []):\n                raise RuntimeError("System chain changed since restore; refusing to overwrite it.")\n        with timer.phase("scan"):\n            current = self.scan()\n        if compact:\n            effective = set()\n            for manifest, _ in self._chain(pointer) if pointer else []:\n                effective.difference_update(manifest["deleted"])\n                effective.update(manifest["files"])\n            changed = {name for name, info in current.items()\n                       if name in effective or info[0] > int(self.start * 1e9)}\n            deleted = set(self.baseline["paths"]) - current.keys()\n            references = []\n        else:\n            def differs(name, info):\n                if name in previous["paths"] and previous["paths"][name] is None:\n                    return info[0] > previous["baseline_before_ns"]\n                return previous["paths"].get(name) != info\n            changed = {name for name, info in current.items() if (\n                differs(name, info) and (pointer is not None or info[0] > int(self.start * 1e9)))}\n            # Initial save includes runtime-created system files even if prepare saw them.\n            if pointer is None:\n                changed.update(name for name, info in current.items() if info[0] > int(self.start * 1e9))\n            deleted = ((set(previous["paths"]) if pointer else set(self.baseline["paths"])) - current.keys())\n            references = (pointer or {}).get("chain", [])\n            if pointer and previous.get("baseline_before_ns") is not None:\n                for name in deleted:\n                    for ancestor in map(str, Path(name).parents):\n                        info = current.get(ancestor)\n                        if info is None:\n                            continue\n                        if (not stat.S_ISDIR(info[3]) and ancestor in previous["paths"]\n                                and previous["paths"][ancestor] is None\n                                and info[0] <= previous["baseline_before_ns"]):\n                            raise RuntimeError("System topology changed before restore: " + ancestor +\n                                               ". Start a fresh VM before restoring and saving.")\n                        break\n        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-" + uuid.uuid4().hex[:8]\n        self.layers.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(dir=self.local) as directory:\n            source = Path(directory) / (stamp + ".tar.gz")\n            with timer.phase("archive"):\n                # pigz uses multiple CPUs when present; gzip is the Python stdlib implementation.\n                if self.real and shutil.which("pigz"):\n                    with source.open("wb") as output:\n                        compressor = subprocess.Popen(["pigz", "-1", "-p", "2"], stdin=subprocess.PIPE, stdout=output)\n                        try:\n                            with tarfile.open(fileobj=compressor.stdin, mode="w|") as archive:\n                                self._archive_files(archive, changed, current)\n                        finally:\n                            compressor.stdin.close()\n                            code = compressor.wait()\n                        if code:\n                            raise RuntimeError("System archive compression failed.")\n                else:\n                    with tarfile.open(source, "w:gz", compresslevel=1) as archive:\n                        self._archive_files(archive, changed, current)\n            with timer.phase("packages"):\n                packages = self._package_changes()\n            manifest = {"schema": 1, "id": stamp, "image_id": self.image_id,\n                        "parent": references[-1]["id"] if references else None,\n                        "sha256": sha256(source), "bytes": source.stat().st_size,\n                        "files": sorted(changed), "deleted": sorted(deleted),\n                        "packages": packages, "created_utc": _utc()}\n            with timer.phase("copy_verify"):\n                self._copy_verified(source, self.layers / source.name, manifest["sha256"])\n        manifest_path = self.layers / (stamp + ".json")\n        write_json(manifest_path, manifest)\n        # Read the manifest back too; publish only after every immutable object is verified.\n        if json.loads(manifest_path.read_text()) != manifest:\n            raise RuntimeError("System manifest readback failed; previous pointer remains current.")\n        result = {"schema": 1, "image_id": self.image_id,\n                  "baseline_sha256": self.baseline_sha256, "created_utc": manifest["created_utc"],\n                  "chain": references + [{"id": stamp, "sha256": sha256(manifest_path)}],\n                  "packages": manifest["packages"], "bytes": manifest["bytes"],\n                  **timer.metrics()}\n        staged = self.local / ("system-state-" + uuid.uuid4().hex + ".json.gz")\n        try:\n            with timer.phase("state_write"):\n                self._write_state(current, result, destination=staged)\n            result.update(timer.metrics())\n            write_json(self.latest, result)\n            os.replace(staged, self.state_path)\n        finally:\n            staged.unlink(missing_ok=True)\n        write_json(self.local / "system-save.json", result)\n        print(f"System saved: {len(changed)} paths, {len(deleted)} deletions, {manifest[\'bytes\']} "\n              f"compressed bytes, {result[\'seconds\']:.2f}s; {len(result[\'chain\'])} layer(s).")\n        print("System save phases (seconds):", json.dumps(result["phases"], sort_keys=True))\n        return result\n\n    def _archive_files(self, archive, names, current):\n        for name in sorted(names):\n            path = self.path(name)\n            before = self.fingerprint(path.lstat(), path)\n            if before != current[name]:\n                raise RuntimeError("System file changed during save: " + name)\n            archive.inodes.clear()  # Store hardlinked files independently; never follow symlinks.\n            archive.add(path, arcname=name.lstrip("/"), recursive=False)\n            if self.fingerprint(path.lstat(), path) != before:\n                raise RuntimeError("System file changed during save: " + name)\n\n    def rebuild(self):\n        """Explicit migration: installers are allowed only through this method."""\n        if self.owner_path.exists() or self.state_path.exists():\n            raise RuntimeError("Use a fresh VM for rebuild-system.")\n        alternatives = sorted((self.root / "environment/system").glob("*/latest.json"),\n                              key=lambda p: json.loads(p.read_text())["created_utc"])\n        if not alternatives:\n            raise RuntimeError("No recorded system packages to rebuild.")\n        recipe = json.loads(alternatives[-1].read_text())["packages"]\n        self._claim()\n        self.ensure_baseline()\n        self._write_state(self.scan(), None)\n        self.migrating = True\n        if recipe["apt"]:\n            subprocess.run(["apt-get", "update", "-qq"], check=True)\n            subprocess.run(["apt-get", "install", "-y", *[name + "=" + version\n                            for name, version in sorted(recipe["apt"].items())]], check=True)\n        if recipe["pip"]:\n            subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",\n                            *[name + "==" + version for name, version in sorted(recipe["pip"].items())]], check=True)\n        return self.save()\n\n\nclass Workspace:\n    """Use durable files on Drive and executable packages on the current VM."""\n\n    def __init__(self, project="default", drive_root=None, local_root=None, session=None,\n                 space="default", system=None):\n        valid_name(project)\n        valid_name(space, "Space")\n        default_drive = Path("/content/drive/MyDrive")\n        if drive_root is None and not default_drive.is_dir():\n            raise RuntimeError("Mount Google Drive before creating a workspace.")\n        self.project = project\n        self.space = space\n        self.session = session\n        # Subprocesses run with cwd=self.root, so relative paths must be absolute.\n        self.data_root = Path(drive_root or default_drive / "PersistentColabData").resolve()\n        self.root = self.data_root / "spaces" / space / "workspaces" / project\n        self.local = (Path(local_root or "/content/persistent-colab") / space / project).resolve()\n        self.venv = self.local / ".venv"\n        self.python = self.venv / "bin" / "python"\n        self.env_dir = self.root / "environment"\n        self.snapshots = self.env_dir / "snapshots"\n        self.latest = self.env_dir / "latest.json"\n        enabled = bool(os.environ.get("COLAB_RELEASE_TAG")) if system is None else system\n        timer = PhaseTimer()\n        with timer.phase("runtime_signature"):\n            self.signature = None if enabled else runtime_signature()\n        self.initialization_metrics = timer.metrics()\n        self.activated = False\n        for folder in ("src", "data", "models", "outputs", "checkpoints", "environment/snapshots"):\n            (self.root / folder).mkdir(parents=True, exist_ok=True)\n        self.local.mkdir(parents=True, exist_ok=True)\n        self.system = SystemLayer(self.root, self.data_root, self.local) if enabled else None\n        # Model files are durable, tokens remain in the ephemeral runtime.\n        os.environ["HF_HUB_CACHE"] = str(self.root / "models" / "huggingface")\n        os.environ["HF_TOKEN_PATH"] = str(self.local / "hf-token")\n        os.environ["TORCH_HOME"] = str(self.root / "models" / "torch")\n\n    def prepare(self, restore=True, force_unlock=False, allow_base_drift=False):\n        """Claim the workspace, recreate executable paths and restore the latest compatible snapshot."""\n        timer = PhaseTimer()\n        with timer.phase("lock"):\n            acquire_lock(self.root, self.session, force_unlock)\n        try:\n            if self.system:\n                with timer.phase("system_restore"):\n                    self.system.restore()\n                with timer.phase("runtime_signature"):\n                    self.signature = runtime_signature()\n            with timer.phase("venv_prepare"):\n                result = self._prepare(restore, allow_base_drift)\n            self.prepare_metrics = {**timer.metrics(), "initialization": self.initialization_metrics}\n            write_json(self.local / "prepare.json", self.prepare_metrics)\n            if self.system:\n                path = self.local / "system-restore.json"\n                if path.exists():\n                    metrics = json.loads(path.read_text())\n                    metrics["prepare"] = self.prepare_metrics\n                    write_json(path, metrics)\n            return result\n        except BaseException:\n            release_lock(self.root)\n            raise\n\n    def _compatibility(self, saved, allow_base_drift):\n        """Return base package drift accepted for this restore, or raise."""\n        rebuild = ("Review environment/latest.json, then explicitly run ws.rebuild() to "\n                   "reinstall the saved requirements on this image. Old snapshots are kept.")\n        if any(saved.get(key) != self.signature.get(key) for key in PLATFORM_KEYS):\n            raise RuntimeError("The Colab Python/base image changed (Python, architecture or libc). "\n                               "The saved environment was not loaded. " + rebuild)\n        if saved.get("base_packages_sha256") == self.signature.get("base_packages_sha256"):\n            return []\n        if "base_packages" not in saved:\n            raise RuntimeError("The Colab Python/base image changed. The saved environment was not loaded. " + rebuild)\n        drift = package_drift(saved["base_packages"], self.signature["base_packages"])\n        if not allow_base_drift:\n            raise RuntimeError("The Colab base image changed; the saved environment was not loaded. "\n                               "Changed base packages:\\n  " + "\\n  ".join(drift) +\n                               "\\nRetry with allow_base_drift=True (CLI: --allow-base-drift) to restore and "\n                               "validate it with pip check, or " + rebuild)\n        print("Base packages differ from the snapshot:\\n  " + "\\n  ".join(drift))\n        return drift\n\n    def _dependency_problems(self, python):\n        result = subprocess.run([str(python), "-m", "pip", "check"], capture_output=True, text=True)\n        return set(result.stdout.splitlines()) if result.returncode else set()\n\n    def _prepare(self, restore, allow_base_drift):\n        ready = self.local / "environment-ready.json"\n        if self.python.exists():\n            if ready.exists() and json.loads(ready.read_text()) == self.signature:\n                return self\n            if self.system and ready.exists() and all(json.loads(ready.read_text()).get(key) ==\n                                                      self.signature.get(key) for key in PLATFORM_KEYS):\n                # System pip installs on this VM intentionally change inherited packages.\n                write_json(ready, self.signature)\n                return self\n            raise RuntimeError("Incomplete local environment. Start a fresh runtime to restore safely.")\n        snapshot = json.loads(self.latest.read_text()) if restore and self.latest.exists() else None\n        drift = self._compatibility(snapshot["runtime"], allow_base_drift) if snapshot else []\n        if snapshot and snapshot["local_path"] != str(self.venv):\n            raise RuntimeError("Snapshot executable paths differ. Use the original project name and runtime path.")\n        # Colab\'s image has pip but its ensurepip bootstrap is unavailable.\n        # Inherit pip from the base image; its install target is still this venv.\n        subprocess.run([sys.executable, "-m", "venv", "--without-pip",\n                        "--system-site-packages", str(self.venv)], check=True)\n        if snapshot:\n            archive_name = snapshot["archive"]\n            if Path(archive_name).name != archive_name:\n                raise ValueError("Invalid snapshot archive name.")\n            source = self.snapshots / archive_name\n            local_archive = self.local / "restore.tar.gz"\n            shutil.copyfile(source, local_archive)\n            if sha256(local_archive) != snapshot["sha256"]:\n                shutil.rmtree(self.venv)\n                local_archive.unlink()\n                raise RuntimeError("Snapshot checksum mismatch; environment was not restored.")\n            try:\n                with tarfile.open(local_archive, "r:gz") as archive:\n                    archive.extractall(self.venv, filter="data")\n            except Exception:\n                # This directory is an agent-owned, disposable VM environment only.\n                shutil.rmtree(self.venv)\n                raise\n            finally:\n                local_archive.unlink(missing_ok=True)\n            if drift:\n                # Colab images already ship some unmet requirements; only new ones fail.\n                problems = self._dependency_problems(self.python) - self._dependency_problems(sys.executable)\n                if problems:\n                    shutil.rmtree(self.venv)\n                    raise RuntimeError("Restored environment conflicts with the new base image; it was removed:\\n  "\n                                       + "\\n  ".join(sorted(problems)))\n                print("pip check found no new conflicts with the changed base image.")\n            print("Restored Python environment:", archive_name)\n        else:\n            print("Created a new Python environment; Colab base packages are available.")\n        write_json(ready, self.signature)\n        print("Persistent workspace:", self.root)\n        return self\n\n    def run(self, code, capture=False):\n        """Run Python in the saved environment; persistent workspace is its cwd."""\n        return subprocess.run([str(self.python), "-c", code], cwd=self.root,\n                              check=True, text=True, capture_output=capture)\n\n    def install(self, *packages):\n        """Install into the environment and immediately persist its new snapshot."""\n        if not packages or any(p.startswith("-") for p in packages):\n            raise ValueError("Pass package specifications, such as \'humanize==4.15.0\'.")\n        subprocess.run([str(self.python), "-m", "pip", "install", *packages], check=True)\n        result = self.save_environment()\n        importlib.invalidate_caches()\n        print("New packages are available in fresh imports; restart before replacing loaded packages.")\n        return result\n\n    def save_environment(self, compact=False):\n        """Commit an immutable archive, publishing the pointer only after readback."""\n        if not self.python.exists():\n            raise RuntimeError("Run ws.prepare() first.")\n        if lock_held_elsewhere(self.root):\n            raise RuntimeError("Another runtime holds this workspace\'s lock; the environment was not saved.")\n        requirements = subprocess.check_output(\n            [str(self.python), "-m", "pip", "freeze", "--local"], text=True)\n        # Do not persist credentials embedded in installation URLs.\n        if "://" in requirements or "-e " in requirements or " @ " in requirements:\n            raise RuntimeError("Use versioned index packages; save local/editable source under ws.root / \'src\'.")\n        if self.system:\n            self.signature = runtime_signature()\n        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-" + uuid.uuid4().hex[:8]\n        name = stamp + ".tar.gz"\n        with tempfile.TemporaryDirectory(dir=self.local) as temporary:\n            archive_path = Path(temporary) / name\n            with tarfile.open(archive_path, "w:gz", compresslevel=1) as archive:\n                for directory in ("lib", "bin", "include"):\n                    source = self.venv / directory\n                    if source.exists():\n                        def exclude_interpreter(info):\n                            if info.name.startswith("bin/python"):\n                                return None\n                            if info.issym() and (self.venv / info.name).resolve() == Path(sys.executable).resolve():\n                                return None\n                            if info.issym() and Path(info.linkname).is_absolute():\n                                raise RuntimeError("Environment contains an external symlink: " + info.name)\n                            if "__pycache__" in Path(info.name).parts:\n                                return None\n                            return info\n                        archive.add(source, arcname=directory, filter=exclude_interpreter)\n            expected = sha256(archive_path)\n            destination = self.snapshots / name\n            with archive_path.open("rb") as source, destination.open("xb") as target:\n                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)\n                target.flush()\n                os.fsync(target.fileno())\n            if sha256(destination) != expected:\n                raise RuntimeError("Drive snapshot readback failed; previous snapshot remains current.")\n        system_snapshot = self.system.save(compact=compact) if self.system else None\n        snapshot = {"schema": 2, "archive": name, "sha256": expected,\n                    "created_utc": stamp, "runtime": self.signature,\n                    "requirements": requirements, "local_path": str(self.venv)}\n        if system_snapshot:\n            snapshot["system"] = system_snapshot\n        write_json(self.snapshots / (stamp + ".json"), snapshot)\n        write_json(self.env_dir / "latest.json", snapshot)\n        print("Environment saved and checksum verified:", name)\n        return snapshot\n\n    def compact(self):\n        """Merge the active system chain; immutable old layers remain until explicit prune."""\n        if self.system is None:\n            raise RuntimeError("System persistence is only available on a Colab VM.")\n        return self.save_environment(compact=True)\n\n    def rebuild_system(self, force_unlock=False):\n        """Explicitly reinstall a recorded system recipe on a fresh, different image."""\n        if self.system is None:\n            raise RuntimeError("System persistence is only available on a Colab VM.")\n        acquire_lock(self.root, self.session, force_unlock)\n        try:\n            result = self.system.rebuild()\n            self.signature = runtime_signature()\n            return result\n        except BaseException:\n            release_lock(self.root)\n            raise\n\n    def rebuild(self, force_unlock=False):\n        """Explicitly reinstall recorded packages when Colab\'s base image changed."""\n        if self.activated:\n            raise RuntimeError("Restart the kernel before rebuilding an activated environment.")\n        previous = json.loads(self.latest.read_text())\n        if self.venv.exists():\n            raise RuntimeError("Use a fresh runtime for rebuilding; current environment is preserved.")\n        self.prepare(restore=False, force_unlock=force_unlock)\n        lock = self.local / "requirements.txt"\n        lock.write_text(previous["requirements"])\n        if previous["requirements"].strip():\n            subprocess.run([str(self.python), "-m", "pip", "install", "-r", str(lock)], check=True)\n        return self.save_environment()\n\n    def activate(self):\n        """Expose this environment to fresh imports in ordinary notebook cells."""\n        paths = json.loads(subprocess.check_output(\n            [str(self.python), "-c", "import site,json; print(json.dumps(site.getsitepackages()))"], text=True))\n        own = [p for p in paths if Path(p).is_relative_to(self.venv)]\n        for path in reversed(own):\n            site.addsitedir(path)\n            sys.path.remove(path)\n            sys.path.insert(0, path)\n        os.environ["PATH"] = str(self.venv / "bin") + os.pathsep + os.environ["PATH"]\n        self.activated = True\n        importlib.invalidate_caches()\n        os.chdir(self.root)\n        print("Notebook imports use the saved environment. Install through ws.install(...).")\n        return self\n\n    def write_checkpoint(self, state, name="session.json"):\n        """Save explicit JSON state; arbitrary RAM/GPU state is never implicit."""\n        if Path(name).name != name or not name.endswith(".json"):\n            raise ValueError("Checkpoint name must be a simple .json filename.")\n        write_json(self.root / "checkpoints" / name, state)\n        return self.root / "checkpoints" / name\n\n    def read_checkpoint(self, name="session.json"):\n        if Path(name).name != name or not name.endswith(".json"):\n            raise ValueError("Checkpoint name must be a simple .json filename.")\n        path = self.root / "checkpoints" / name\n        return json.loads(path.read_text()) if path.exists() else None\n\n    def flush(self):\n        """Flush storage writes before the user releases the VM."""\n        release_lock(self.root)\n        state = rclone_state()\n        if state and self.root.is_relative_to(Path(state["mount_point"]).resolve()):\n            flush_rclone([self.root])\n        else:\n            from google.colab import drive\n            drive.flush_and_unmount()\n        print("Storage writes flushed. You may now disconnect and delete this runtime.")\n')
ws = Workspace(PROJECT, space=SPACE)
ws.prepare()
ws.activate()
print('Previous saved state:', ws.read_checkpoint())


## Work here

Your current directory is the durable Drive workspace. Use `ws.root / 'src'`, `'data'`, `'models'`, `'outputs'`, and `'checkpoints'` for files that must survive.

Install workspace Python packages with `ws.install('package==version')`; it saves the environment immediately. The environment inherits Colab's preinstalled AI libraries. `%pip` and `!pip` affect the system environment instead; save those changes with `ws.save_environment()`. Use `ws.run(code)` or `subprocess.run([str(ws.python), ...])` for an independent Python process and for packages already imported by the notebook kernel. Installing a replacement for an already loaded package requires restarting the kernel and running cells 1 and 2 again.

Hugging Face model downloads and Torch downloads use the Drive model directory. Authentication tokens stay on the temporary runtime. Keep secrets in Colab Secrets; never in the notebook, workspace or checkpoints.

For many small files, unpack a dataset on the runtime's local disk for speed, retaining its original archive on Drive. Only files actually saved back to Drive survive. Use one project per VM: system restoration changes the whole VM. Spaces and projects separate directories, not account permissions. Concurrent writers to the same project are unsupported.


In [ ]:
# @title Optional: install and save packages
# Uncomment and change the package list when needed.
# ws.install('humanize==4.15.0')


In [ ]:
# @title Your Python / AI code
from pathlib import Path
print('Working directory:', Path.cwd())
print('Persistent output directory:', ws.root / 'outputs')
print('Environment Python:', ws.python)
# Write your code in this cell or add more cells here.


## Save progress explicitly

Use `ws.write_checkpoint({'step': 100, ...})` for JSON state; load it with `ws.read_checkpoint()`. This is application-defined state, not an automatic copy of Python variables.

For Hugging Face Trainer, set `output_dir=str(ws.root / 'checkpoints' / 'training')`, save checkpoints regularly, and use `trainer.train(resume_from_checkpoint=...)` to resume a complete checkpoint. Saving model weights alone does not preserve optimizer, scheduler and training progress. Keep the model, dataset and training configuration compatible when resuming.

`ws.save_environment()` also saves apt/system pip installation state and filesystem changes under `/usr/local`, `/opt`, `/etc`, `/var/lib` and `/root/.local`. It excludes volatile paths, known credential locations and other mounts. Keep secrets in Colab Secrets; arbitrary secrets placed in captured directories can still be archived. Configuration files may persist, but running OS services and host GPU drivers do not.


In [ ]:
# @title Save environment
saved_environment = ws.save_environment()
print('Saved at:', saved_environment['created_utc'])


## Before releasing compute

Check **FLUSH_DRIVE** and run the next cell **only when finished**. It releases the workspace lock, then flushes and unmounts Drive. Then disconnect and delete the runtime using the Runtime menu. To continue working after unmounting, mount Drive again.

If Colab reports a system/base-image mismatch, start a fresh runtime, mount Drive and instantiate `ws` from cell 2. Run `ws.rebuild_system()` to explicitly reinstall recorded apt/system pip packages and start a new system layer chain; arbitrary files from the old image are not replayed onto a different image. Then use `ws.rebuild()` for saved virtual-environment package requirements and `ws.activate()`. Keep required manual setup in source control. Reinstallation can fail when packages no longer support the new image; old snapshots remain on Drive.

`ws.compact()` replaces the current system layer chain with a compact layer; old layers remain until an explicit prune. CPU and GPU runtimes use different base images. `ws.prepare(allow_base_drift=True)` only relaxes virtual-environment base-package checks with `pip check`; it does not bypass the system image check.

Cell 2 claims the workspace for this runtime. If another runtime ended without flushing, the error reports a stale lock after 10 minutes; once you are sure that runtime is gone, use `ws.prepare(force_unlock=True)`.


In [ ]:
# @title Flush Drive (run manually when finished)
FLUSH_DRIVE = False  # @param {type:"boolean"}
if FLUSH_DRIVE:
    ws.flush()
else:
    print('Check FLUSH_DRIVE and run this cell when you are finished.')
